# Vision-Language Model (VLM) with PyTorch and Transformers
 
**Summary:** This notebook demonstrates the construction, training, and evaluation of a Vision-Language Model (VLM) for image captioning. The model combines a Vision Transformer (ViT) as an image encoder and a Llama-based language model as a text decoder. The pipeline includes dataset preparation, model configuration, training, and inference. All steps are explained in detail with technical and mathematical context.
 
## Implementation Plan
1. **Dataset Preparation:** Download and preprocess image-caption pairs, convert images to base64, and organize data in a DataFrame.
2. **Reproducibility:** Set seeds for deterministic results across Python, NumPy, and PyTorch.
3. **Configuration:** Define all hyperparameters and model settings for both ViT and Llama components.
4. **Tokenizer Setup:** Load and configure the Llama tokenizer for text processing.
5. **Image Preprocessing:** Implement functions to convert images to base64 and back to tensors, with normalization.
6. **Vision Transformer (ViT) Setup:** Configure and test the image encoder for extracting visual features.
7. **Model Architecture:** Build a custom VLM class that fuses ViT and Llama, projecting image embeddings to the language model's space and concatenating them with token embeddings.
8. **Dataset Class:** Implement a PyTorch Dataset to load (image, caption) pairs and prepare them for training.
9. **Training Loop:** Train the model using cross-entropy loss, validate periodically, and monitor loss.
10. **Inference:** Generate captions for new images using the trained model.
 
## Mathematical Formulation
Given an image $I$ and its caption $C = (w_1, w_2, ..., w_n)$, the model learns $P(C|I)$, i.e., the probability of the caption given the image. The loss is the cross-entropy between the predicted and true tokens:
$$
\mathcal{L} = -\sum_{t=1}^n \log P(w_t | I, w_{<t})
$$
The ViT encodes $I$ to a vector $v_I$, which is projected and concatenated with the token embeddings of $C$ before being fed to the Llama decoder.
 
---

In [1]:
!git clone https://huggingface.co/datasets/uygarkurt/simple-image-captions

Cloning into 'simple-image-captions'...
Filtering content:  12% (2/16)
Filtering content:  18% (3/16)
Filtering content:  25% (4/16)
Filtering content:  31% (5/16), 746.42 KiB | 919.00 KiB/s
Filtering content:  37% (6/16), 746.42 KiB | 919.00 KiB/s
Filtering content:  43% (7/16), 746.42 KiB | 919.00 KiB/s
Filtering content:  50% (8/16), 746.42 KiB | 919.00 KiB/s
Filtering content:  56% (9/16), 746.42 KiB | 919.00 KiB/s
Filtering content:  62% (10/16), 1.38 MiB | 846.00 KiB/s 
Filtering content:  68% (11/16), 1.38 MiB | 846.00 KiB/s
Filtering content:  75% (12/16), 1.38 MiB | 846.00 KiB/s
Filtering content:  81% (13/16), 1.38 MiB | 846.00 KiB/s
Filtering content:  87% (14/16), 1.38 MiB | 846.00 KiB/s
Filtering content:  93% (15/16), 3.30 MiB | 794.00 KiB/s
Filtering content: 100% (16/16), 3.30 MiB | 794.00 KiB/s
Filtering content: 100% (16/16), 8.34 MiB | 513.00 KiB/s, done.


## Step 1: Dataset Acquisition
 
**Purpose:** Download a simple image-caption dataset from HuggingFace for use in VLM training and evaluation.
 
**Technical Details:**
- Uses `git clone` to fetch the dataset repository containing images and a CSV file with captions.
- The dataset is small and suitable for rapid prototyping and demonstration.
 
**Why:**
- Provides a ready-to-use, labeled dataset for supervised learning of image-to-text mappings.

In [2]:
import base64
import io
import pandas as pd
import torch
import torchvision.transforms as transforms
import torch.nn as nn
from transformers import LlamaConfig, LlamaForCausalLM, LlamaTokenizer
from transformers import ViTConfig, ViTMAEConfig, ViTModel
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import numpy as np

## Step 2: Import Libraries and Dependencies
 
**Purpose:** Import all required Python libraries for data processing, model building, and training.
 
**Technical Details:**
- `base64`, `io`: For image encoding/decoding.
- `pandas`: Data manipulation and CSV handling.
- `torch`, `torchvision`, `nn`: PyTorch core and vision utilities.
- `transformers`: HuggingFace models (ViT, Llama, Tokenizer).
- `tqdm`: Progress bars for training loops.
- `random`, `numpy`: Randomness and numerical operations.
 
**Why:**
- Ensures all necessary tools are available for the VLM pipeline, from data loading to model training and evaluation.

In [3]:
# For repilcation

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Step 3: Reproducibility Setup
 
**Purpose:** Ensure deterministic and reproducible results across runs.
 
**Technical Details:**
- Sets seeds for Python's `random`, NumPy, and PyTorch.
- Configures PyTorch's backend for deterministic behavior (disables certain optimizations for reproducibility).
 
**Why:**
- Guarantees that experiments can be repeated with the same results, which is critical for debugging and scientific reporting.

In [4]:
BATCH_SIZE = 16
N_HIDDEN_LAYERS = 16
MAX_LENGTH = 16
EVAL_INTERVAL = 10
LEARNING_RATE = 9e-4
EPOCHS = 6
N_EMBD = 128
N_HEAD = 8
N_LAYER = 8
DROPOUT = 0.4
IMG_SIZE = 96
PATCH_SIZE = 16
IMAGE_EMBED_DIM = 512
N_CHANNELS = 3
MAX_POSITION_EMBEDDINGS = 128

## Step 4: Model and Training Configuration
 
**Purpose:** Define all hyperparameters and configuration settings for the VLM pipeline.
 
**Technical Details:**
- **BATCH_SIZE**: Number of samples per batch during training.
- **N_HIDDEN_LAYERS**: Number of transformer layers in the ViT encoder.
- **MAX_LENGTH**: Maximum length of tokenized captions.
- **EVAL_INTERVAL**: Steps between validation checks during training.
- **LEARNING_RATE**: Learning rate for the Adam optimizer.
- **EPOCHS**: Number of full passes over the training data.
- **N_EMBD**: Embedding dimension for the Llama language model.
- **N_HEAD**: Number of attention heads in transformers.
- **N_LAYER**: Number of transformer layers in Llama.
- **DROPOUT**: Dropout probability for regularization.
- **IMG_SIZE**: Input image size (pixels).
- **PATCH_SIZE**: Patch size for ViT.
- **IMAGE_EMBED_DIM**: Output dimension of ViT encoder.
- **N_CHANNELS**: Number of image channels (RGB=3).
- **MAX_POSITION_EMBEDDINGS**: Maximum sequence length for Llama.
 
**Why:**
- Centralizes all configuration for easy tuning and reproducibility.

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = LlamaTokenizer.from_pretrained("NousResearch/Llama-2-7b-chat-hf")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## Step 5: Device and Tokenizer Setup
 
**Purpose:** Set up the computation device (CPU/GPU) and initialize the Llama tokenizer for text processing.
 
**Technical Details:**
- `device`: Automatically selects CUDA (GPU) if available, otherwise falls back to CPU.
- `LlamaTokenizer`: Loads a pre-trained tokenizer for the Llama language model.
- Sets padding and end-of-sequence tokens for consistent batching and decoding.
 
**Why:**
- Ensures efficient computation and proper text tokenization for the language model component of the VLM pipeline.

In [12]:
import os

image_dir = (
    "F:\\PyTorch_GPU\\VLM-pytorch_fundamentals\\Notebooks\\simple-image-captions\\"
)


def image_file_to_base64(image_filename):
    image_path = os.path.join(image_dir, image_filename)
    with open(image_path, "rb") as img_file:
        b64_str = base64.b64encode(img_file.read()).decode("utf-8")
    return b64_str


df = pd.read_csv(os.path.join(image_dir, "inputs.csv"), sep=";").dropna(
    axis=1, how="all"
)
df["b64string_images"] = df["file"].apply(image_file_to_base64)
df.head()

,file,caption,b64string_images
0,car.png,red car,iVBORw0KGgoAAAANSUhEUgAABgAAAAQACAIAAACoEwUVAA...
1,astronaut.png,astronaut in a white space suit,iVBORw0KGgoAAAANSUhEUgAABAAAAAYACAIAAABn4K39AA...
2,tv.png,black television on a table,iVBORw0KGgoAAAANSUhEUgAABgAAAAQACAIAAACoEwUVAA...
3,horse.png,brown horse running,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
4,wine.png,wine bottle,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...


## Step 6: Image Directory, Preprocessing, and DataFrame Construction
 
**Purpose:** Prepare image data for model input by converting images to base64 strings and loading captions into a DataFrame.
 
**Technical Details:**
- `image_dir`: Path to the directory containing images and CSV file.
- `image_file_to_base64`: Converts image files to base64-encoded strings for efficient DataFrame storage.
- Loads the CSV file with image filenames and captions, drops empty columns, and adds a column of base64-encoded images.
 
**Why:**
- Facilitates efficient storage and retrieval of image data for training and evaluation.
- Prepares the data in a format compatible with PyTorch Dataset and DataLoader.

In [13]:
config = ViTConfig(
    image_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    num_channels=N_CHANNELS,
    hidden_size=IMAGE_EMBED_DIM,
    num_attention_heads=N_HEAD,
    num_hidden_layers=N_HIDDEN_LAYERS,
    intermediate_size=4 * IMAGE_EMBED_DIM,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT,
)


testvit = ViTModel(config=config)
vit_input = torch.zeros(BATCH_SIZE, N_CHANNELS, IMG_SIZE, IMG_SIZE)
testvit_output = testvit(vit_input).last_hidden_state[:, 0]
testvit_output.shape

torch.Size([16, 512])

## Step 7: Vision Transformer (ViT) Configuration and Testing
 
**Purpose:** Configure the ViT image encoder and verify its output shape for a dummy input batch.
 
**Technical Details:**
- `ViTConfig`: Sets up the Vision Transformer with parameters matching the configuration section.
- `ViTModel`: Instantiates the ViT encoder for extracting visual features from images.
- Tests the encoder with a zero tensor to ensure correct output dimensions:
  - Input: $[\text{BATCH\_SIZE}, \text{N\_CHANNELS}, \text{IMG\_SIZE}, \text{IMG\_SIZE}]$
  - Output: $[\text{BATCH\_SIZE}, \text{IMAGE\_EMBED\_DIM}]$ (CLS token)
 
**Why:**
- Validates the image encoder setup before integrating with the language model.

In [14]:
class VisionLanguageModel(nn.Module):
    def __init__(
        self,
        n_embed,
        image_embed_dim,
        vocab_size,
        n_layer,
        n_head,
        img_size,
        patch_size,
        n_hidden_layers,
        dropout,
        pad_token_id,
        max_position_embeddings,
        n_channels,
    ):
        super().__init__()
        vit_config = ViTConfig(
            image_size=img_size,
            patch_size=patch_size,
            num_channels=n_channels,
            hidden_size=image_embed_dim,
            num_attention_heads=n_head,
            num_hidden_layers=n_hidden_layers,
            intermediate_size=4 * image_embed_dim,
            hidden_dropout_prob=dropout,
            attention_probs_dropout_prob=dropout,
        )
        self.vision_encoder = ViTModel(vit_config)
        self.image_projector = nn.Linear(image_embed_dim, n_embed)

        llama_config = LlamaConfig(
            vocab_size=vocab_size,
            hidden_size=n_embed,
            num_hidden_layers=n_layer,
            num_attention_heads=n_head,
            max_position_embeddings=max_position_embeddings,
            pad_token_id=int(pad_token_id),
        )
        self.llama = LlamaForCausalLM(llama_config)
        self.llama = self.llama.to(dtype=torch.bfloat16)  # Move Llama to bfloat16

    def forward(self, img_array, input_ids, targets=None):
        # img_array: [BATCH_SIZE, N_CHANNELS, IMG_SIZE, IMG_SIZE]
        # input_ids: [BATCH_SIZE, MAX_LENGTH]
        image_embeds = self.vision_encoder(img_array).last_hidden_state[
            :, 0
        ]  # [BATCH_SIZE, IMAGE_EMBED_DIM]
        image_embeds_proj = self.image_projector(image_embeds).to(
            dtype=torch.bfloat16
        )  # [BATCH_SIZE, N_EMBED]
        image_embeds_proj = image_embeds_proj.unsqueeze(1)  # [BATCH_SIZE, 1, N_EMBED]

        text_embeds = self.llama.model.embed_tokens(input_ids).to(
            dtype=torch.bfloat16
        )  # [BATCH_SIZE, MAX_LENGTH, N_EMBED]

        input_embeds = torch.cat(
            [image_embeds_proj, text_embeds], dim=1
        )  # [BATCH_SIZE, MAX_LENGTH + 1, N_EMBED]

        attention_mask = torch.ones(
            input_embeds.shape[:2], dtype=torch.long, device=input_embeds.device
        )  # [BATCH_SIZE, MAX_LENGTH + 1]

        if targets is not None:
            # target: [BATCH_SIZE, MAX_LENGTH]
            targets = torch.cat(
                [
                    torch.full(
                        (targets.size(0), 1),
                        -100,
                        dtype=targets.dtype,
                        device=targets.device,
                    ),
                    targets,
                ],
                dim=1,
            )  # [BATCH_SIZE, MAX_LENGTH + 1]
            outputs = self.llama(
                inputs_embeds=input_embeds,
                attention_mask=attention_mask,
                labels=targets,
            )
            return outputs.logits, outputs.loss
        else:
            outputs = self.llama(
                inputs_embeds=input_embeds,
                attention_mask=attention_mask,
            )
            return outputs.logits

    @torch.no_grad()
    def generate(self, img_array, input_ids, max_new_tokens=20):
        # img_array: [BATCH_SIZE, N_CHANNELS, IMG_SIZE, IMG_SIZE]
        # input_ids: [BATCH_SIZE, MAX_LENGTH]
        image_embeds = self.vision_encoder(img_array).last_hidden_state[:, 0]
        image_embeds_proj = (
            self.image_projector(image_embeds).unsqueeze(1).to(dtype=torch.bfloat16)
        )

        input_embeds = self.llama.model.embed_tokens(input_ids).to(dtype=torch.bfloat16)
        inputs_embeds = torch.cat([image_embeds_proj, input_embeds], dim=1)
        attention_mask = torch.ones(
            inputs_embeds.shape[:2], dtype=torch.long, device=inputs_embeds.device
        )

        generated = self.llama.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            pad_token_id=self.llama.config.pad_token_id,
            eos_token_id=self.llama.config.eos_token_id,
        )
        return generated


model = VisionLanguageModel(
    N_EMBD,
    IMAGE_EMBED_DIM,
    tokenizer.vocab_size,
    N_LAYER,
    N_HEAD,
    IMG_SIZE,
    PATCH_SIZE,
    N_HIDDEN_LAYERS,
    DROPOUT,
    tokenizer.pad_token_id,
    max_position_embeddings=MAX_POSITION_EMBEDDINGS,
    n_channels=N_CHANNELS,
)
model.to(device)

dummy_img = torch.randn(1, N_CHANNELS, IMG_SIZE, IMG_SIZE).to(device)
dummy_idx = torch.randint(0, tokenizer.vocab_size, (1, MAX_LENGTH)).to(device)
output = model(dummy_img, dummy_idx)
print(output.shape)

torch.Size([1, 17, 32000])


## Step 8: Vision-Language Model (VLM) Architecture
 
**Purpose:** Define a custom PyTorch module that fuses the ViT image encoder and Llama language model for multimodal learning.
 
**Technical Details:**
- **ViT Encoder:** Extracts image features as a fixed-size vector.
- **Image Projector:** Projects ViT output to match Llama's embedding dimension.
- **Llama Decoder:** Processes tokenized captions, concatenated with projected image embeddings.
- **Forward Pass:**
  - Concatenates image and text embeddings: $[\text{image\_embeds\_proj}, \text{text\_embeds}]$
  - Applies attention mask and computes logits/loss via Llama.
- **Generation:**
  - Enables autoregressive caption generation given an image and prompt.
- **Mathematical Formulation:**
  - The model learns $P(C|I)$, where $C$ is the caption and $I$ is the image.
  - Loss: $\mathcal{L} = -\sum_{t=1}^n \log P(w_t | I, w_{<t})$
 
**Why:**
- Enables end-to-end training for image captioning by combining visual and textual modalities in a single architecture.

In [17]:
from PIL import Image

img_path = "F:\\PyTorch_GPU\\VLM-pytorch_fundamentals\\Notebooks\\simple-image-captions\\tree.png"
image = Image.open(img_path).convert("RGB")
transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
img_tensor = (
    transform(image).unsqueeze(0).to(device)
)  # Shape: [BATCH_SIZE (1), N_CHANNELS, IMG_SIZE, IMG_SIZE]

prompt = "A photo of"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    generated_ids = model.generate(img_tensor, input_ids, max_new_tokens=30)
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("Generated description of the given picture:")
print(generated_text)

Generated description of the given picture:
initialize світfunction highestraitarto DAT.$$ certificate play internet писа писа писа尔 ю star baimatelypredjunitAPI raggroupId одинOrientation Rund Referências erh debut


## Step 9: Image Captioning Demo (Single Image Inference)
 
**Purpose:** Demonstrate the VLM's ability to generate a caption for a single image using the trained model pipeline.
 
**Technical Details:**
- Loads and preprocesses a sample image using PIL and torchvision transforms.
- Tokenizes a prompt and runs the model's `generate` method to produce a caption.
- Decodes the generated token IDs to human-readable text.
 
**Why:**
- Provides a quick sanity check and qualitative evaluation of the model's image-to-text capabilities before full training.

In [19]:
## helper function
from PIL import Image
import base64
import io
import torchvision.transforms as transforms


def base64_to_tensor(b64_string, img_size):
    img_bytes = base64.b64decode(b64_string)
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    transform = transforms.Compose(
        [
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    return transform(img).unsqueeze(0)

## Step 10: Helper Function for Base64 Image Decoding
 
**Purpose:** Define a utility function to convert base64-encoded image strings back to normalized PyTorch tensors for model input.
 
**Technical Details:**
- Decodes base64 string to bytes, loads image with PIL, and applies resizing, normalization, and tensor conversion.
- Ensures images are in the correct format and scale for the ViT encoder.
 
**Why:**
- Enables efficient storage of images in DataFrames and seamless conversion for model training and inference.

In [20]:
class VLMDataset(Dataset):
    def __init__(self, df, img_size=96, tokenizer=None):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_b64 = self.df.loc[idx, "b64string_images"]
        caption = self.df.loc[idx, "caption"]
        image = base64_to_tensor(img_b64, self.img_size).squeeze(0)
        encoding = self.tokenizer(
            caption,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
        )
        input_ids = encoding.input_ids.squeeze(0)
        targets = input_ids.clone()
        targets[:-1] = input_ids[1:]
        targets[-1] = self.tokenizer.pad_token_id
        return image, input_ids, targets


df = pd.concat([df] * 50)[["b64string_images", "caption"]]
n = int(0.9 * len(df))
df_train = df.iloc[:n]
df_val = df.iloc[n:]

train_dataset = VLMDataset(df_train, img_size=IMG_SIZE, tokenizer=tokenizer)
val_dataset = VLMDataset(df_val, img_size=IMG_SIZE, tokenizer=tokenizer)

print(len(train_dataset))
print(len(train_dataset[0]))

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, drop_last=True)

36000
3


## Step 11: Custom PyTorch Dataset for VLM
 
**Purpose:** Implement a Dataset class to efficiently load and preprocess (image, caption) pairs for training and validation.
 
**Technical Details:**
- Loads base64-encoded images and captions from the DataFrame.
- Converts images to tensors and tokenizes captions to input/target IDs.
- Prepares data for teacher-forcing during training (shifts targets by one position).
- Splits the dataset into training and validation sets.
 
**Why:**
- Enables efficient batching and shuffling for model training using PyTorch DataLoader.
- Ensures each batch contains properly formatted image and text data.

In [21]:
@torch.no_grad()
def estimate_loss(model, val_loader):
    losses = []
    model.eval()
    for images, input_ids, targets in val_loader:
        images = images.to(device)
        input_ids = input_ids.to(device)
        targets = targets.to(device)
        _, loss = model(images, input_ids, targets)
        losses.append(loss.item())
    return sum(losses) / len(losses)

## Step 12: Validation Loss Estimation Function
 
**Purpose:** Define a function to estimate the average validation loss over the validation set, used for model selection and monitoring.
 
**Technical Details:**
- Disables gradient computation for efficiency (`@torch.no_grad()`).
- Iterates over the validation DataLoader, computes loss for each batch, and averages the results.
- Moves data to the correct device (CPU/GPU) before inference.
 
**Why:**
- Provides a quantitative measure of model generalization and helps prevent overfitting.

In [22]:
def train_model(model, train_loader, val_loader, epochs, learning_rate, eval_interval):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    for epoch in range(epochs):
        model.train()
        loop = tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            desc=f"Epoch {epoch+1}/{epochs}",
        )
        for batch_idx, (images, input_ids, targets) in loop:
            images = images.to(device)
            input_ids = input_ids.to(device)
            targets = targets.to(device)
            optimizer.zero_grad()
            logits, loss = model(images, input_ids, targets)
            loss.backward()
            optimizer.step()
            if batch_idx % eval_interval == 0:
                loop.set_postfix(loss=loss.item())
        val_loss = estimate_loss(model, val_loader)
        print(f"Validation Loss after epoch {epoch}: {val_loss}")

## Step 13: Model Training Loop
 
**Purpose:** Train the VLM using the Adam optimizer, periodically evaluating on the validation set and reporting loss.
 
**Technical Details:**
- Iterates over epochs and batches, performing forward and backward passes.
- Updates model parameters using gradients from cross-entropy loss.
- Prints training loss and evaluates validation loss at specified intervals.
- Uses `tqdm` for progress visualization.
 
**Why:**
- Enables the model to learn the mapping from images to captions through supervised learning.

In [ ]:
train_model(model, train_loader, val_loader, EPOCHS, LEARNING_RATE, EVAL_INTERVAL)

Epoch 1/6: 100%|██████████| 2250/2250 [18:14<00:00,  2.06it/s, loss=0.121] 


Validation Loss after epoch 0: 0.1849433295428753


Epoch 2/6: 100%|██████████| 2250/2250 [21:49<00:00,  1.72it/s, loss=0.174] 


Validation Loss after epoch 1: 0.3604508936405182


Epoch 3/6: 100%|██████████| 2250/2250 [23:24<00:00,  1.60it/s, loss=0.122] 


Validation Loss after epoch 2: 0.28343820571899414


Epoch 4/6: 100%|██████████| 2250/2250 [22:56<00:00,  1.63it/s, loss=0.165] 


Validation Loss after epoch 3: 0.2357478141784668


Epoch 5/6:  21%|██        | 471/2250 [04:54<19:06,  1.55it/s, loss=0.169] 

## Step 14: Start Model Training
 
**Purpose:** Launch the training process for the Vision-Language Model using the defined training loop and configuration.
 
**Technical Details:**
- Calls the `train_model` function with the model, DataLoaders, and hyperparameters.
- Training progress and validation loss are printed for each epoch.
 
**Why:**
- Initiates the learning process, allowing the model to optimize its parameters for the image captioning task.

In [ ]:
img_path = "F:\\PyTorch_GPU\\VLM-pytorch_fundamentals\\Notebooks\\simple-image-captions\\tree.png"
image = Image.open(img_path).convert("RGB")
transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
img_tensor = transform(image).unsqueeze(0).to(device)

prompt = "A photo of"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

model.eval()
with torch.no_grad():
    generated_ids = model.generate(img_tensor, input_ids, max_new_tokens=30)
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("Generated description of the given picture:")

## Step 15: Inference on a New Image
 
**Purpose:** Use the trained VLM to generate a caption for a new, unseen image, demonstrating the model's inference pipeline.
 
**Technical Details:**
- Loads and preprocesses a test image.
- Tokenizes a prompt and generates a caption using the model's `generate` method.
- Decodes the output tokens to produce a human-readable description.
 
**Why:**
- Validates the model's ability to generalize and perform image-to-text generation on new data.